# Experiment: AKI Models

Objective:
- Load the cleaned AKI cohort.
- Build leakage-aware feature matrices with a subject-level split.
- Train simple baseline models and report AUROC / AUPRC.


In [ ]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, classification_report, roc_auc_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

INPUT_CSV = Path('outputs/mimic_aki_cohort_cleaned.csv')
TARGET_COL = 'future_aki_24h'
ID_COLS = ['subject_id', 'hadm_id', 'stay_id']
TIME_COLS = [
    'icu_intime',
    'anchor_time',
    'obs_window_start',
    'obs_window_end',
    'pred_window_start',
    'pred_window_end',
]
TEST_SIZE = 0.20
RANDOM_STATE = 42


## Load cleaned dataset

This notebook assumes `aki_data_cleaning.ipynb` has already exported `outputs/mimic_aki_cohort_cleaned.csv`.


In [ ]:
df = pd.read_csv(INPUT_CSV)
print(f'Cleaned shape: {df.shape[0]:,} rows x {df.shape[1]:,} columns')
print('Class balance:')
print(df[TARGET_COL].value_counts(dropna=False).sort_index())
print('Unique subjects:', df['subject_id'].nunique())
df.head()


## Target and feature column review

Identifiers and timing metadata stay in the dataframe for auditability, but they must not be passed into the model. Any obvious outcome-style columns are excluded as a second leakage guard.


In [ ]:
extra_leakage_cols = [
    col for col in [
        'hospital_expire_flag',
        'los_icu',
        'los_hospital',
        'icu_outtime',
        'dischtime',
        'aki_onset_time',
        'aki_onset_stage',
        'aki_stage_max',
        'has_aki_anytime',
        'eligible_for_prediction',
    ]
    if col in df.columns
]

excluded_feature_cols = set(ID_COLS + TIME_COLS + [TARGET_COL] + extra_leakage_cols)
feature_cols = [col for col in df.columns if col not in excluded_feature_cols]

print(f'Feature count: {len(feature_cols)}')
print('Excluded columns:', sorted(excluded_feature_cols))
feature_preview = pd.DataFrame({'feature': feature_cols})
feature_preview.head(20)


## Subject-level train/test split

Rows from the same patient must not be split across train and test. We use `GroupShuffleSplit` with `subject_id` as the group key.


In [ ]:
groups = df['subject_id']
gss = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_STATE)
train_idx, test_idx = next(gss.split(df, df[TARGET_COL], groups=groups))

train_df = df.iloc[train_idx].reset_index(drop=True)
test_df = df.iloc[test_idx].reset_index(drop=True)

print({'train_rows': len(train_df), 'test_rows': len(test_df)})
print({'train_subjects': train_df['subject_id'].nunique(), 'test_subjects': test_df['subject_id'].nunique()})
print('Overlapping subjects:', len(set(train_df['subject_id']).intersection(set(test_df['subject_id']))))


## Preprocessing pipeline

Numeric features use median imputation plus scaling for logistic regression. Categorical features use most-frequent imputation and one-hot encoding.


In [ ]:
X_train = train_df[feature_cols].copy()
X_test = test_df[feature_cols].copy()
y_train = train_df[TARGET_COL].astype(int)
y_test = test_df[TARGET_COL].astype(int)

numeric_cols = X_train.select_dtypes(include=['number', 'bool']).columns.tolist()
categorical_cols = [col for col in feature_cols if col not in numeric_cols]


def make_one_hot_encoder():
    try:
        return OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown='ignore', sparse=False)

preprocessor = ColumnTransformer(
    transformers=[
        (
            'numeric',
            Pipeline([
                ('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler()),
            ]),
            numeric_cols,
        ),
        (
            'categorical',
            Pipeline([
                ('imputer', SimpleImputer(strategy='most_frequent')),
                ('onehot', make_one_hot_encoder()),
            ]),
            categorical_cols,
        ),
    ],
    remainder='drop',
)

print({'numeric_features': len(numeric_cols), 'categorical_features': len(categorical_cols)})


## Logistic regression baseline


In [ ]:
logistic_model = Pipeline([
    ('preprocess', preprocessor),
    ('model', LogisticRegression(max_iter=1000, class_weight='balanced')),
])
logistic_model.fit(X_train, y_train)

logistic_proba = logistic_model.predict_proba(X_test)[:, 1]
logistic_pred = (logistic_proba >= 0.5).astype(int)

logistic_metrics = {
    'model': 'logistic_regression',
    'auroc': roc_auc_score(y_test, logistic_proba),
    'auprc': average_precision_score(y_test, logistic_proba),
}
logistic_metrics


In [ ]:
print(classification_report(y_test, logistic_pred, digits=3))


## Random forest baseline


In [ ]:
rf_model = Pipeline([
    ('preprocess', preprocessor),
    ('model', RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        min_samples_leaf=2,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        class_weight='balanced_subsample',
    )),
])
rf_model.fit(X_train, y_train)

rf_proba = rf_model.predict_proba(X_test)[:, 1]
rf_pred = (rf_proba >= 0.5).astype(int)

rf_metrics = {
    'model': 'random_forest',
    'auroc': roc_auc_score(y_test, rf_proba),
    'auprc': average_precision_score(y_test, rf_proba),
}
rf_metrics


In [ ]:
print(classification_report(y_test, rf_pred, digits=3))


## Optional boosted tree baseline

The import is guarded so the notebook still runs if XGBoost is unavailable in Colab.


In [ ]:
optional_metrics = None
try:
    from xgboost import XGBClassifier

    xgb_model = Pipeline([
        ('preprocess', preprocessor),
        ('model', XGBClassifier(
            n_estimators=300,
            max_depth=4,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            eval_metric='logloss',
            random_state=RANDOM_STATE,
        )),
    ])
    xgb_model.fit(X_train, y_train)
    xgb_proba = xgb_model.predict_proba(X_test)[:, 1]
    optional_metrics = {
        'model': 'xgboost',
        'auroc': roc_auc_score(y_test, xgb_proba),
        'auprc': average_precision_score(y_test, xgb_proba),
    }
except ImportError:
    print('XGBoost is not installed in this Colab runtime; skipping optional boosted-tree baseline.')

optional_metrics


## Metrics summary

Compare baseline models on the same subject-level holdout split.


In [ ]:
metrics_rows = [logistic_metrics, rf_metrics]
if optional_metrics is not None:
    metrics_rows.append(optional_metrics)

metrics_df = pd.DataFrame(metrics_rows).sort_values('auprc', ascending=False)
metrics_df


## Next steps

- Tune thresholds after you inspect calibration and class balance.
- Add cross-validation at the subject level if the first baseline looks stable.
- If feature sparsity is severe, revisit `aki_data_cleaning.ipynb` before adding model complexity.
